# Custom operators and units

You can define your own operator set and a unit system for a new domain. An
operator bundles a numeric function, a SymPy function, a domain-validity
predicate, and a unit rule.

In [ ]:
import numpy as np
import jax.numpy as jnp
from jax_ibart import engineer
from jax_ibart.operators import Operator, BINARY

# Inspect a built-in operator
op = BINARY["mul"]
op.name, op.arity, op.commutative

### Units: a worked dimensional example

Units are integer-exponent vectors over base dimensions. Additive operators only
combine *matching* units; multiply/divide add/subtract exponents. Here feature
`x` is a length and `z` is a time, so `x + z` is rejected but `x*z` and `x/z` are
allowed.

In [ ]:
X = np.random.default_rng(0).uniform(1, 2, size=(12, 2))
units = np.array(
    [
        [1, 0],  # x: [length, time] exponents
        [0, 1],
    ]
)  # z
pool = engineer.from_primary(X, ["x", "z"], units=units)

add = engineer.apply_binary(pool, [BINARY["add"]])
mul = engineer.apply_binary(pool, [BINARY["mul"]])
print("x + z generated:", add.n_descriptors, "(rejected: mismatched units)")
print("x * z generated:", mul.n_descriptors, "-> unit", mul.units[0].tolist())

### Defining your own operator

A cube operator: numeric `x**3`, symbolic `e**3`, valid everywhere, unit `3*u`.

In [ ]:
cube = Operator(
    name="cube",
    arity=1,
    fn=lambda x: x**3,
    sym=lambda e: e**3,
    valid=lambda x: jnp.ones(x.shape[-1], bool),
    unit=lambda u: 3 * u,
)
out = engineer.apply_unary(pool, [cube])
print([str(e) for e in out.exprs], "-> units", out.units.tolist())

Pass custom operators by editing the operator sets used in `IBart.fit`, or use
them directly with the `engineer` module as shown. The same lockstep design
holds: numbers live in JAX arrays, symbols in SymPy, units in integer vectors.